# 序号27：绩效归因分析

## 学习目标
- 掌握 **Brinson 归因**：将超额收益拆解为配置效应、选择效应、交互效应
- 掌握 **因子归因**：用 Fama-French 模型拆解收益来源（市场、规模、价值等）
- 区分 **择时能力 vs 选股能力**
- 输出一份像样的归因报告

## 验收标准
- ✅ 能拆解策略的收益来源
- ✅ 理解运气和能力的区别
- ✅ 能写一份像样的归因报告

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import akshare as ak
import warnings, time
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('✅ 环境就绪')

## 0. 数据准备

获取沪深300成分股的分行业数据和 Fama-French 因子数据。

In [ ]:
# 获取沪深300成分股
print('📡 获取沪深300成分股...')
hs300 = ak.index_stock_cons_csindex(symbol='000300')
stocks_all = hs300['成分券代码'].tolist()
print(f'共 {len(stocks_all)} 只成分股')

# 取前20只大市值股票做演示（减少API调用量）
sample_codes = stocks_all[:20]
print(f'演示用: {len(sample_codes)}只')

# 获取行业分类
print('\n📡 获取行业分类...')
industry_map = {}
for i, code in enumerate(sample_codes):
    try:
        info = ak.stock_individual_info_em(symbol=code)
        if info is not None and len(info) > 0:
            d = dict(zip(info['item'], info['value']))
            industry_map[code] = d.get('行业', '未知')
    except: 
        industry_map[code] = '未知'
    time.sleep(0.1)

print(f'已获取 {len(industry_map)} 只股票的行业分类')
industry_s = pd.Series(industry_map).value_counts()
print(f'\n行业分布 (前10):\n{industry_s.head(10)}')

In [ ]:
# 获取价格数据（近1年）
print('📡 获取价格数据...')
price_data = {}
for i, code in enumerate(sample_codes):
    try:
        d = ak.stock_zh_a_hist(symbol=code, period='daily',
                               start_date='20250101', end_date='20260622',
                               adjust='qfq')
        d['日期'] = pd.to_datetime(d['日期'])
        d = d.set_index('日期').sort_index()
        if len(d) > 50:
            price_data[code] = d['收盘']
    except: pass
    if (i+1) % 10 == 0: print(f'  进度: {i+1}/{len(sample_codes)}')
    time.sleep(0.15)

price_df = pd.DataFrame(price_data)
ret_df = price_df.pct_change().dropna()
print(f'\n价格数据: {price_df.shape}')
print(f'日期范围: {price_df.index[0].date()} ~ {price_df.index[-1].date()}')

## 1. 构建演示策略组合

我们模拟一个"主动管理组合"：在某些行业超配、某些行业低配，与沪深300等权基准对比。
然后对这个组合做归因——拆解超额收益的来源。

In [ ]:
# 构建行业 → 股票映射
industry_stocks = {}
for code, ind in industry_map.items():
    if code in ret_df.columns:
        industry_stocks.setdefault(ind, []).append(code)

# 合并小行业为"其它"
major_industries = []
for ind, codes in industry_stocks.items():
    if len(codes) >= 2:
        major_industries.append(ind)
    else:
        industry_stocks.setdefault('其它', []).extend(codes)
        if ind != '其它': del industry_stocks[ind]

print(f'行业数: {len(industry_stocks)}')
for ind, codes in sorted(industry_stocks.items(), key=lambda x: -len(x[1])):
    print(f'  {ind}: {len(codes)}只 {codes}')

# === 基准：等权组合 ===
benchmark_ret = ret_df.mean(axis=1)

# === 主动组合：模拟"看好银行、低配科技"的主动偏离 ===
# 给每个行业分配权重
total_stocks = sum(len(v) for v in industry_stocks.values())
active_weights = {}

weight_multipliers = {}  # 行业权重乘数
for ind in industry_stocks:
    if '银行' in ind:
        weight_multipliers[ind] = 1.8  # 超配
    elif '保险' in ind or '证券' in ind:
        weight_multipliers[ind] = 1.3  # 小幅超配
    elif '电子' in ind or '计算机' in ind or '半导体' in ind:
        weight_multipliers[ind] = 0.5  # 低配
    else:
        weight_multipliers[ind] = 1.0  # 中性

# 计算个股权重
raw_weights = {}
for ind, codes in industry_stocks.items():
    for c in codes:
        if c in ret_df.columns:
            raw_weights[c] = weight_multipliers.get(ind, 1.0) / len(codes)

# 归一化
total_w = sum(raw_weights.values())
portfolio_weights = {k: v/total_w for k, v in raw_weights.items()}

# 主动组合日收益
active_ret = pd.Series(0.0, index=ret_df.index)
for code, w in portfolio_weights.items():
    if code in ret_df.columns:
        active_ret += w * ret_df[code]

print(f'\n===== 组合对比 =====')
cum_active = (1 + active_ret).cumprod()
cum_bench = (1 + benchmark_ret).cumprod()
print(f'主动组合累计收益: {cum_active.iloc[-1] - 1:.2%}')
print(f'基准累计收益:     {cum_bench.iloc[-1] - 1:.2%}')
print(f'超额收益:         {(cum_active.iloc[-1] - cum_bench.iloc[-1]):.2%}')

excess_ret = active_ret - benchmark_ret
tracking_error = excess_ret.std() * np.sqrt(252)
info_ratio = excess_ret.mean() / excess_ret.std() * np.sqrt(252) if excess_ret.std() > 0 else 0
print(f'年化跟踪误差: {tracking_error:.2%}')
print(f'信息比率:     {info_ratio:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_active.index, cum_active.values, color='#2563eb', linewidth=2, label='主动组合')
ax.plot(cum_bench.index, cum_bench.values, color='gray', linewidth=1.5, alpha=0.7, label='等权基准')
ax.fill_between(cum_active.index, cum_active.values, cum_bench.values,
                 alpha=0.15, color='green' if cum_active.iloc[-1] > cum_bench.iloc[-1] else 'red',
                 label='超额收益' if cum_active.iloc[-1] > cum_bench.iloc[-1] else '超额亏损')
ax.set_title('主动组合 vs 基准', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---

## 2. Brinson 归因

### 2.1 原理

Brinson 模型将超额收益分解为三个部分：

$$R_P - R_B = \underbrace{\sum (w_{P,i} - w_{B,i}) \cdot R_{B,i}}_{\text{配置效应 Allocation}} + \underbrace{\sum w_{B,i} \cdot (R_{P,i} - R_{B,i})}_{\text{选择效应 Selection}} + \underbrace{\sum (w_{P,i} - w_{B,i}) \cdot (R_{P,i} - R_{B,i})}_{\text{交互效应 Interaction}}$$

| 效应 | 含义 | 正面解读 |
|------|------|----------|
| **配置效应** | 超配表现好的行业？ | 择时/行业轮动能力 |
| **选择效应** | 行业内选的股票跑赢行业平均？ | 选股能力 |
| **交互效应** | 超配的行业内选股也更好？ | 双重的叠加效果 |

In [ ]:
# === Brinson 归因计算 ===

# 1. 计算各行业基准收益（行业内等权）和主动组合在行业内的收益
industry_bench_ret = {}  # 行业基准收益
industry_active_ret = {} # 主动组合在各行业的收益
industry_bench_weight = {}  # 行业基准权重
industry_active_weight = {} # 主动组合行业权重

for ind, codes in industry_stocks.items():
    valid_codes = [c for c in codes if c in ret_df.columns]
    if len(valid_codes) == 0:
        continue
    
    # 行业基准收益：行业内等权
    ind_ret = ret_df[valid_codes].mean(axis=1)
    industry_bench_ret[ind] = ind_ret.mean() * 252  # 年化
    
    # 行业基准权重：等权
    industry_bench_weight[ind] = len(valid_codes) / sum(len(v) for v in industry_stocks.values() if any(c in ret_df.columns for c in v))
    
    # 主动组合行业权重
    active_w = sum(portfolio_weights.get(c, 0) for c in valid_codes)
    industry_active_weight[ind] = active_w
    
    # 主动组合在行业内的收益
    total_w_in_ind = sum(portfolio_weights.get(c, 0) for c in valid_codes)
    if total_w_in_ind > 0:
        weighted_ret = pd.Series(0.0, index=ret_df.index)
        for c in valid_codes:
            weighted_ret += (portfolio_weights.get(c, 0) / total_w_in_ind) * ret_df[c]
        industry_active_ret[ind] = weighted_ret.mean() * 252
    else:
        industry_active_ret[ind] = 0

# 总基准收益和主动收益
R_B = sum(industry_bench_weight[ind] * industry_bench_ret[ind] for ind in industry_bench_weight)
R_P = sum(industry_active_weight[ind] * industry_active_ret.get(ind, 0) for ind in industry_active_weight)

print(f'基准年化收益 R_B = {R_B:.2%}')
print(f'主动年化收益 R_P = {R_P:.2%}')
print(f'超额收益 = {R_P - R_B:.2%}\n')

In [ ]:
# 计算三个效应
allocation_effect = {}
selection_effect = {}
interaction_effect = {}

for ind in industry_bench_weight:
    wB = industry_bench_weight[ind]
    wP = industry_active_weight.get(ind, 0)
    rB = industry_bench_ret[ind]
    rP = industry_active_ret.get(ind, rB)
    
    allocation_effect[ind] = (wP - wB) * rB
    selection_effect[ind] = wB * (rP - rB)
    interaction_effect[ind] = (wP - wB) * (rP - rB)

# 汇总
total_alloc = sum(allocation_effect.values())
total_select = sum(selection_effect.values())
total_interact = sum(interaction_effect.values())
total_excess = total_alloc + total_select + total_interact

print('===== Brinson 归因结果 =====')
print(f'配置效应 (Allocation):   {total_alloc:+.2%}')
print(f'选择效应 (Selection):    {total_select:+.2%}')
print(f'交互效应 (Interaction):  {total_interact:+.2%}')
print(f'─' * 25)
print(f'总超额收益:              {total_excess:+.2%}')
print(f'验证 (R_P - R_B):       {R_P - R_B:+.2%}')

# 行业明细
attr_df = pd.DataFrame({
    '行业权重差': {k: f'{industry_active_weight.get(k,0)-industry_bench_weight.get(k,0):+.1%}' for k in allocation_effect},
    '配置效应': {k: f'{v:+.2%}' for k, v in allocation_effect.items()},
    '选择效应': {k: f'{v:+.2%}' for k, v in selection_effect.items()},
    '交互效应': {k: f'{v:+.2%}' for k, v in interaction_effect.items()},
}).sort_values('配置效应', ascending=False)
print('\n===== 行业明细 =====')
print(attr_df)

In [ ]:
# Brinson 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：瀑布图
ax = axes[0]
effects = ['配置效应', '选择效应', '交互效应', '总超额']
values = [total_alloc, total_select, total_interact, total_excess]
colors_bar = ['#2563eb' if v > 0 else '#dc2626' for v in values]
bars = ax.bar(effects, [v*100 for v in values], color=colors_bar, alpha=0.85, edgecolor='white')
ax.axhline(y=0, color='gray', linestyle='-')
for bar, v in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.05*np.sign(v),
            f'{v:+.2%}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Brinson 归因瀑布图', fontsize=13, fontweight='bold')
ax.set_ylabel('超额收益贡献 (%)')
ax.grid(True, alpha=0.3, axis='y')

# 右：行业级配置vs选择
ax = axes[1]
inds = list(allocation_effect.keys())[:8]
x = np.arange(len(inds))
w = 0.35
alloc_vals = [allocation_effect[i]*100 for i in inds]
select_vals = [selection_effect[i]*100 for i in inds]
ax.bar(x-w/2, alloc_vals, w, color='#2563eb', alpha=0.8, label='配置效应')
ax.bar(x+w/2, select_vals, w, color='#f59e0b', alpha=0.8, label='选择效应')
ax.set_xticks(x)
ax.set_xticklabels([i[:4] for i in inds], fontsize=8)
ax.axhline(y=0, color='gray', linestyle='-')
ax.set_title('行业级归因明细', fontsize=13, fontweight='bold')
ax.set_ylabel('贡献 (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

---

## 3. 因子归因

### 3.1 原理

用 Fama-French 五因子模型回归组合超额收益：

$$R_P - R_f = \alpha + \beta_{MKT} \cdot MKT + \beta_{SMB} \cdot SMB + \beta_{HML} \cdot HML + \beta_{RMW} \cdot RMW + \beta_{CMA} \cdot CMA + \varepsilon$$

- **α（Alpha）**：无法被因子解释的超额收益 → **真正的选股/择时能力**
- **β 系数**：组合对各因子的暴露 → 收益有多少来自"承担了某种风险"

如果 α 显著为正，说明你确实有技能。如果 β 解释了大部分收益，说明你只是"搭了某种风格的顺风车"。

In [ ]:
# 获取 Fama-French 五因子日度数据
print('📡 获取 Fama-French 五因子数据...')

# 使用 akshare 获取
try:
    ff5 = ak.fama_french_5factor_daily(
        start_date='20250101', end_date='20260622'
    )
except:
    # 备选：用沪深300相关因子
    ff5 = ak.index_factor_em(symbol='000300')

print(f'FF5数据: {ff5.shape if ff5 is not None else "获取失败"}')

# 处理因子数据
if ff5 is not None:
    if '日期' in ff5.columns or 'trade_date' in ff5.columns:
        date_col = '日期' if '日期' in ff5.columns else 'trade_date'
        ff5[date_col] = pd.to_datetime(ff5[date_col])
        ff5 = ff5.set_index(date_col).sort_index()
    
    # 标准化列名
    col_map = {}
    for c in ff5.columns:
        cl = c.lower()
        if 'mkt' in cl or 'market' in cl: col_map[c] = 'MKT'
        elif 'smb' in cl: col_map[c] = 'SMB'
        elif 'hml' in cl: col_map[c] = 'HML'
        elif 'rmw' in cl: col_map[c] = 'RMW'
        elif 'cma' in cl: col_map[c] = 'CMA'
        elif 'rf' in cl or 'riskfree' in cl: col_map[c] = 'RF'
    ff5 = ff5.rename(columns=col_map)
    
    factor_cols = [c for c in ['MKT', 'SMB', 'HML', 'RMW', 'CMA'] if c in ff5.columns]
    print(f'可用因子: {factor_cols}')
    print(ff5[factor_cols].tail())
else:
    factor_cols = []

In [ ]:
# 因子归因回测
if len(factor_cols) >= 3:
    # 对齐日期
    common_dates = excess_ret.index.intersection(ff5.index)
    
    y = excess_ret.loc[common_dates] * 100  # 转为百分比
    X = ff5.loc[common_dates, factor_cols]
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    print('===== Fama-French 因子归因 =====')
    print(model.summary().tables[1])
    
    alpha_daily = model.params.get('const', 0)
    alpha_annual = alpha_daily * 252
    alpha_t = model.tvalues.get('const', 0)
    alpha_p = model.pvalues.get('const', 0)
    
    print(f'\n===== Alpha 解读 =====')
    print(f'日度 Alpha: {alpha_daily:.4f}%')
    print(f'年化 Alpha: {alpha_annual:.2f}%')
    print(f't 统计量:  {alpha_t:.2f}')
    print(f'p 值:      {alpha_p:.4f}')
    print(f'R²:        {model.rsquared:.4f}')
    
    if alpha_p < 0.05 and alpha_annual > 0:
        print('✅ 存在显著正 Alpha → 有真正的选股/择时能力')
    elif alpha_p < 0.05 and alpha_annual < 0:
        print('❌ 存在显著负 Alpha → 策略在"创造"负价值')
    else:
        print('⚠️ Alpha 不显著 → 超额收益可被因子完全解释（搭了顺风车）')
else:
    print('⚠️ 因子数据不足，跳过因子归因')

In [ ]:
if len(factor_cols) >= 3:
    # 可视化因子暴露
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左：因子暴露柱状图（含置信区间）
    ax = axes[0]
    betas = {}
    cis = {}
    for col in factor_cols:
        betas[col] = model.params[col]
        cis[col] = (model.conf_int().loc[col, 0], model.conf_int().loc[col, 1])
    
    cols_plot = list(betas.keys())
    vals = [betas[c] for c in cols_plot]
    err_low = [betas[c] - cis[c][0] for c in cols_plot]
    err_high = [cis[c][1] - betas[c] for c in cols_plot]
    
    colors_f = ['#2563eb' if v > 0 else '#dc2626' for v in vals]
    ax.bar(cols_plot, vals, color=colors_f, alpha=0.85, edgecolor='white',
           yerr=[err_low, err_high], capsize=5)
    ax.axhline(y=0, color='gray', linestyle='-')
    for i, (c, v) in enumerate(zip(cols_plot, vals)):
        p = model.pvalues[c]
        sig_mark = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        ax.text(i, v + 0.02*np.sign(v), f'{v:.3f}{sig_mark}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title('因子暴露 (Beta)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # 右：收益分解饼图
    ax = axes[1]
    # 各因子的收益贡献 = beta × 因子均值
    factor_contrib = {}
    for col in factor_cols:
        factor_contrib[col] = model.params[col] * ff5.loc[common_dates, col].mean() * 252
    factor_contrib['Alpha'] = alpha_annual
    
    # 只显示绝对值较大的
    contrib_s = pd.Series(factor_contrib).sort_values(key=abs, ascending=False)
    
    colors_pie = ['#2563eb' if v > 0 else '#dc2626' for v in contrib_s.values]
    wedges, texts, autotexts = ax.pie(
        [abs(v) for v in contrib_s.values], 
        labels=[f'{k}\n{v:+.2f}%' for k, v in contrib_s.items()],
        colors=colors_pie, autopct='', startangle=90,
        explode=[0.05]*len(contrib_s)
    )
    ax.set_title('年化收益来源分解', fontsize=13, fontweight='bold')
    
    plt.tight_layout(); plt.show()
else:
    print('⚠️ 跳过因子归因可视化')

---

## 4. 择时能力 vs 选股能力

### 4.1 Henriksson-Merton 模型

$$R_P - R_f = \alpha + \beta_1 \cdot (R_M - R_f) + \beta_2 \cdot \max(R_M - R_f, 0) + \varepsilon$$

- $\beta_2 > 0$ 且显著 → 有**择时能力**（市场涨时 β 更高，市场跌时 β 更低）
- $\alpha > 0$ 且显著 → 有**选股能力**

### 4.2 Treynor-Mazuy 模型

$$R_P - R_f = \alpha + \beta_1 \cdot (R_M - R_f) + \beta_2 \cdot (R_M - R_f)^2 + \varepsilon$$

- $\beta_2 > 0$ → 有择时能力（回报曲线向上弯曲）

In [ ]:
# Henriksson-Merton 择时模型
if len(factor_cols) >= 1:
    common_dates = excess_ret.index.intersection(ff5.index)
    
    # 用 MKT 因子作为市场收益
    mkt_factor = ff5.loc[common_dates, 'MKT'] if 'MKT' in ff5.columns else ff5.loc[common_dates, factor_cols[0]]
    
    y = excess_ret.loc[common_dates] * 100
    
    # Henriksson-Merton: max(Rm, 0) 表示市场上涨时的暴露
    mkt_up = np.maximum(mkt_factor, 0)
    
    X_hm = pd.DataFrame({
        'MKT': mkt_factor,
        'MKT_up': mkt_up
    })
    X_hm = sm.add_constant(X_hm)
    
    model_hm = sm.OLS(y, X_hm).fit()
    
    print('===== Henriksson-Merton 择时模型 =====')
    print(model_hm.summary().tables[1])
    
    gamma = model_hm.params.get('MKT_up', 0)
    gamma_p = model_hm.pvalues.get('MKT_up', 1)
    alpha_hm = model_hm.params.get('const', 0)
    alpha_hm_p = model_hm.pvalues.get('const', 1)
    
    print(f'\n===== 解读 =====')
    if gamma > 0 and gamma_p < 0.1:
        print(f'择时能力 γ = {gamma:.4f} (p={gamma_p:.4f}) → ✅ 有市场择时能力')
    else:
        print(f'择时能力 γ = {gamma:.4f} (p={gamma_p:.4f}) → ❌ 无显著择时能力')
    
    if alpha_hm > 0 and alpha_hm_p < 0.1:
        print(f'选股能力 α = {alpha_hm:.4f}% (p={alpha_hm_p:.4f}) → ✅ 有选股能力')
    else:
        print(f'选股能力 α = {alpha_hm:.4f}% (p={alpha_hm_p:.4f}) → ❌ 无显著选股能力')
else:
    print('⚠️ 因子数据不足')

In [ ]:
# 可视化：CAPM 特征线 vs 实际收益
if len(factor_cols) >= 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    common_dates = excess_ret.index.intersection(ff5.index)
    mkt = ff5.loc[common_dates, 'MKT'] if 'MKT' in ff5.columns else ff5.loc[common_dates, factor_cols[0]]
    y_vals = excess_ret.loc[common_dates] * 100
    
    # 左：散点图 + 特征线
    ax = axes[0]
    ax.scatter(mkt, y_vals, alpha=0.3, s=10, c='steelblue')
    # 拟合线
    X_simple = sm.add_constant(mkt)
    model_simple = sm.OLS(y_vals, X_simple).fit()
    x_range = np.linspace(mkt.min(), mkt.max(), 100)
    y_pred = model_simple.params['const'] + model_simple.params.iloc[1] * x_range
    ax.plot(x_range, y_pred, 'r-', linewidth=2, label=f'β={model_simple.params.iloc[1]:.2f}')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.5)
    ax.set_xlabel('市场因子 (MKT) %', fontsize=11)
    ax.set_ylabel('超额收益 %', fontsize=11)
    ax.set_title('CAPM 特征线', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 右：滚动 Alpha（检验稳定性）
    ax = axes[1]
    window = min(60, len(y_vals) // 3)
    rolling_alpha = []
    rolling_dates = []
    for i in range(window, len(y_vals)):
        y_win = y_vals.iloc[i-window:i]
        X_win = sm.add_constant(mkt.iloc[i-window:i])
        m = sm.OLS(y_win, X_win).fit()
        rolling_alpha.append(m.params.iloc[0] * 252)  # 年化
        rolling_dates.append(y_vals.index[i])
    
    ax.plot(rolling_dates, rolling_alpha, color='#2563eb', linewidth=1.5)
    ax.axhline(y=0, color='gray', linestyle='-')
    ax.fill_between(rolling_dates, 0, rolling_alpha, alpha=0.15, color='blue')
    ax.set_title(f'滚动年化 Alpha ({window}日窗口)', fontsize=13, fontweight='bold')
    ax.set_ylabel('年化 Alpha')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout(); plt.show()

---

## 5. 归因报告模板

一份像样的归因报告至少包含以下内容：

In [ ]:
# 生成归因报告摘要
print('='*60)
print('           绩 效 归 因 报 告')
print('='*60)
print(f'报告日期: 2026-06-23')
print(f'回测期间: {ret_df.index[0].date()} ~ {ret_df.index[-1].date()}')
print()
print('【组合概况】')
print(f'  主动组合年化收益: {R_P:.2%}')
print(f'  基准年化收益:     {R_B:.2%}')
print(f'  超额收益:         {R_P - R_B:+.2%}')
print(f'  年化跟踪误差:     {tracking_error:.2%}')
print(f'  信息比率:         {info_ratio:.2f}')
print()
print('【Brinson 归因】')
print(f'  配置效应:   {total_alloc:+.2%}', '← 行业超配/低配的贡献' if total_alloc > 0 else '← 行业配置拖了后腿')
print(f'  选择效应:   {total_select:+.2%}', '← 行业内选股的贡献' if total_select > 0 else '')
print(f'  交互效应:   {total_interact:+.2%}')
print()

# 找出最大贡献和最大拖累的行业
best_ind = max(allocation_effect, key=allocation_effect.get)
worst_ind = min(allocation_effect, key=allocation_effect.get)
print(f'  最大配置贡献: {best_ind} ({allocation_effect[best_ind]:+.2%})')
print(f'  最大配置拖累: {worst_ind} ({allocation_effect[worst_ind]:+.2%})')
print()

if len(factor_cols) >= 3:
    print('【因子归因 (FF5)】')
    print(f'  Alpha (年化): {alpha_annual:+.2%}', 
          '✅ 显著' if alpha_p < 0.05 else '(不显著)')
    print(f'  因子解释度 R²: {model.rsquared:.2%}')
    for col in factor_cols:
        b = model.params[col]
        p = model.pvalues[col]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        print(f'  β_{col}: {b:+.4f}{sig}')
    print()
else:
    print('【因子归因】数据不足，跳过\n')

print('【结论与建议】')

if total_alloc > 0:
    print('  ✅ 行业配置能力正面，建议保持行业轮动框架')
else:
    print('  ⚠️ 行业配置拖累收益，建议审视超低配逻辑')

if total_select > 0:
    print('  ✅ 行业内选股能力正面')
else:
    print('  ⚠️ 选股能力不足，建议优化个股选择标准')

if len(factor_cols) >= 3:
    if alpha_annual > 0 and alpha_p < 0.1:
        print('  ✅ 存在正的选股 Alpha（因子不可解释的部分），继续深挖')
    else:
        print('  ⚠️ 超额收益主要来自因子暴露（风格 Beta），非独特技能')
else:
    pass

print()
print('='*60)
print('免责声明：本报告仅用于学习演示，不构成投资建议。')
print('='*60)

---

## 6. 总结

### 三种归因方法对比

| 方法 | 核心问题 | 输出 | 适用场景 |
|------|----------|------|----------|
| **Brinson** | 收益来自配置还是选股？ | 配置/选择/交互效应 | 多行业组合 |
| **因子归因** | 收益是 Alpha 还是 Beta？ | α + 各因子贡献 | 任何组合 |
| **H-M / T-M** | 是择时好还是选股好？ | 择时 γ + 选股 α | 灵活仓位组合 |

### 核心洞察

> **"你的超额收益是能力还是运气？"——归因分析就是回答这个问题。**

- Brinson 说："你超配的行业表现好 ≠ 你有眼光，可能只是碰巧"
- 因子归因说："你的 Alpha 是 3% 但 p=0.6 → 这 3% 大概率是噪声"
- H-M 模型说："你既不会择时也不会选股，但你不承认"

### 验收自检

- [x] 能拆解策略的收益来源（Brinson 三效应）
- [x] 理解运气和能力的区别（Alpha 的统计显著性）
- [x] 能写一份像样的归因报告

> **下一步**：序号28 — 策略研究报告（综合实战项目），整合策略→回测→风控→归因全流程。